In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

orders = pd.read_csv('/content/olist_orders_dataset.csv')
reviews = pd.read_csv('/content/olist_order_reviews_dataset.csv')

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [4]:
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

delivered = orders.dropna(subset=['order_delivered_customer_date', 'order_estimated_delivery_date']).copy()
delivered['is_late'] = delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']

delivered['is_late'].value_counts()

,count
is_late,
False,88649
True,7827


In [5]:
merged = delivered.merge(reviews[['order_id', 'review_score']], on='order_id', how='inner')
merged = merged.dropna(subset=['review_score'])

print(len(merged), 'orders with both delivery status and a review')
merged[['order_id', 'is_late', 'review_score']].head()

103387 orders with both delivery status and a review


,order_id,is_late,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,False,4
1,e481f51cbdc54678b7cc49136f2d6af7,False,4
2,53cdb2fc8bc7dce0b6741e2150273451,False,4
3,53cdb2fc8bc7dce0b6741e2150273451,False,4
4,47770eb9100c2d0c44946d9cf07ec65d,False,5


In [6]:
# Check how many order_ids have duplicate review rows
print('Total rows:', len(merged))
print('Unique order_ids:', merged['order_id'].nunique())

# Look at one duplicated example
merged[merged['order_id'] == merged['order_id'].iloc[0]]

Total rows: 103387
Unique order_ids: 95830


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_late,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,4
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False,4


In [7]:
merged = merged.drop_duplicates(subset=['order_id', 'review_score'])
print('After removing duplicates:', len(merged))
print('Unique order_ids:', merged['order_id'].nunique())

After removing duplicates: 96019
Unique order_ids: 95830


In [8]:
# Find orders that still have more than one row
dupe_orders = merged[merged.duplicated(subset='order_id', keep=False)]
print('Orders with multiple different review scores:', dupe_orders['order_id'].nunique())
dupe_orders.sort_values('order_id')[['order_id', 'review_score']].head(10)

Orders with multiple different review scores: 189


,order_id,review_score
87548,013056cfe49763c6f66bda03396c5ee3,4
87547,013056cfe49763c6f66bda03396c5ee3,5
83743,02355020fd0a40a0d56df9f6ff060413,1
83744,02355020fd0a40a0d56df9f6ff060413,3
76296,029863af4b968de1e5d6a82782e662f5,4
76295,029863af4b968de1e5d6a82782e662f5,5
12909,03c939fd7fd3b38f8485a0f95798f1f6,3
12910,03c939fd7fd3b38f8485a0f95798f1f6,4
45041,03eba6d9fef8f5b3e811d4b5a7cca9cd,5
45040,03eba6d9fef8f5b3e811d4b5a7cca9cd,4


In [9]:
merged = merged.drop_duplicates(subset='order_id', keep='first')
print('Final row count:', len(merged))
print('Unique order_ids:', merged['order_id'].nunique())

Final row count: 95830
Unique order_ids: 95830


In [13]:
merged['review_score'] = pd.to_numeric(merged['review_score'], errors='coerce')
merged.groupby('is_late')['review_score'].describe()

,count,mean,std,min,25%,50%,75%,max
is_late,,,,,,,,
False,88168.0,4.294358,1.147060,1.0,4.0,5.0,5.0,5.0
True,7662.0,2.565518,1.658425,1.0,1.0,2.0,4.0,5.0


In [14]:
on_time_scores = merged[merged['is_late'] == False]['review_score']
late_scores = merged[merged['is_late'] == True]['review_score']

t_stat, p_value = stats.ttest_ind(on_time_scores, late_scores, equal_var=False)
print(f'T-test: t={t_stat:.3f}, p={p_value:.6f}')

u_stat, p_value_mw = stats.mannwhitneyu(on_time_scores, late_scores, alternative='two-sided')
print(f'Mann-Whitney U: U={u_stat:.1f}, p={p_value_mw:.6f}')

print('Mean review score (on-time):', on_time_scores.mean().round(2))
print('Mean review score (late):', late_scores.mean().round(2))

T-test: t=89.410, p=0.000000
Mann-Whitney U: U=524812980.5, p=0.000000
Mean review score (on-time): 4.29
Mean review score (late): 2.57


 **6. Interpretation**

Orders delivered on time received an average review score of 4.29 out of 5,
while late orders averaged just 2.57 — a drop of nearly 1.7 stars. Both the
t-test (t=89.41) and the Mann-Whitney U test produced p-values reported as
0.000000 (i.e., far smaller than 0.05, effectively p < 0.0001).

In plain language: this difference is not due to random chance. With a
sample this large (88,168 on-time vs. 7,662 late orders), even a small
true difference would likely be statistically significant — but a gap of
nearly two full stars is also a large, practically meaningful effect, not
just a statistically detectable one. Customers who experience a late
delivery are, on average, dramatically less satisfied, and their ratings
are also far more spread out (std = 1.66 vs 1.15), suggesting some late
customers are still forgiving while many are very unhappy.

**7. Limitations**
- **Correlation, not causation**: lateness is strongly associated
with lower scores, but this analysis doesn't prove lateness alone causes
  the drop — damaged items, poor product quality, or bad customer
  service could occur alongside delays and drive dissatisfaction too.
- **Large sample size inflates statistical significance**: with 95,830
  orders, even a very small true difference would likely produce a
  tiny p-value. The p-value alone doesn't tell us the size of the
  effect — that's why the ~1.7-star mean difference matters more than
  the p-value itself here.
- **"Late" is binary**: a delivery 1 day late and one 30 days late are
  treated identically. The actual severity of delay isn't captured.
- **Excluded orders**: orders missing a delivery date, estimated date,
  or review were dropped. If those orders differ systematically
  (e.g., never delivered = automatically low satisfaction), excluding
  them could understate the true effect.
- **Multiple reviews per order**: 189 orders had more than one review
  score; only the first was kept, which is a simplification.

**8. Conclusion**

The evidence strongly supports the conclusion that late deliveries are
associated with substantially lower customer review scores on Olist —
on-time orders average 4.29 stars vs. 2.57 for late orders, a difference
confirmed by two independent statistical tests (p < 0.0001).

**What I can conclude**: delivery timing and customer satisfaction (as
measured by review score) are strongly related in this dataset, and the difference is both statistically significant and practically large.

**What I cannot conclude**: that late delivery is the sole or direct
cause of lower ratings, since other factors correlated with delays
(e.g., product issues, seller reliability) weren't controlled for. I also cannot claim this relationship holds equally across all product categories, regions, or price points, since the analysis didn't segment
by those variables. A reasonable business takeaway is that reducing
delivery delays is likely to meaningfully improve customer satisfaction,
but confirming delivery timing as the primary driver would require
further analysis that isolates it from other variables.

**What I Learned**

This project taught me how to move from a simple question to a statistically
grounded answer. I learned that choosing the right test matters — since
review scores are skewed and ordinal, pairing a t-test with a Mann-Whitney
U test gave a more honest picture than either alone. I also learned that a
tiny p-value with a huge sample size isn't automatically a "big" finding;
what made this result meaningful was the actual size of the gap (4.29 vs
2.57 stars), not just its statistical significance. Cleaning the data —
catching duplicate and conflicting reviews before they silently skewed my
results — was just as important as the analysis itself. Finally, I learned
to be explicit about what my analysis can and can't prove: it shows a
strong association between late delivery and low satisfaction, but not
that lateness alone causes it.

Name:- Wadud Ansari


Email:- wadudansari2002@gmail.com or erwadudansari@gmail.com